# SupplyMind Environment Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShAuRyA-Noodle/Sleep-Token/blob/main/notebooks/01_environment_quickstart.ipynb)

This notebook demonstrates how to interact with the SupplyMind environment — a Gymnasium-compatible RL environment for supply chain risk management.

**Time:** <10 minutes on CPU

In [ ]:
# Bootstrap so `import rl` / `from server...` resolve on Colab (git clone) and local Jupyter (sys.path).
import os, sys, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/ShAuRyA-Noodle/Sleep-Token.git'
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    ROOT = Path('/content/Sleep-Token')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    # Local Jupyter: notebook lives in notebooks/, so the repo root is the parent dir.
    ROOT = Path.cwd()
    if not (ROOT / 'rl').exists() and (ROOT.parent / 'rl').exists():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Repo root on sys.path: {ROOT}  (Colab={IN_COLAB})')


In [ ]:
# Install dependencies (Colab)
!pip install -q gymnasium==0.29.1 networkx numpy pydantic fastapi

In [ ]:
import gymnasium as gym
import numpy as np
import rl  # Register environments

# Create the easy task environment
env = gym.make('SupplyMind-Easy-v1')
print(f'Observation space: {env.observation_space}')
print(f'Action space: {env.action_space}')

In [ ]:
# Reset and inspect initial state
obs, info = env.reset(seed=42)
print(f'Observation shape: {obs.shape}')
print(f'Action mask shape: {info["action_masks"].shape}')
print(f'Valid actions: {info["action_masks"].sum()} / 280')
print(f'\nFirst 10 obs values: {obs[:10]}')

In [ ]:
# Run a random episode
obs, info = env.reset(seed=42)
total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    if terminated or truncated:
        break

print(f'Episode finished in {steps} steps')
print(f'Total reward: {total_reward:.4f}')

In [ ]:
# Run the scripted agent
from server.supply_environment import SupplyMindEnvironment
from scripted_agent import choose_action, run_task

env_core = SupplyMindEnvironment()
result = run_task(env_core, 'easy_typhoon_response', seed=42)
print(f'Score: {result["score"]:.4f}')
print(f'Steps: {result["steps_taken"]}')
for k, v in result.get('breakdown', {}).items():
    print(f'  {k}: {v["score"]:.4f} (weight={v["weight"]})')

In [ ]:
# Compare all 3 tasks
tasks = ['easy_typhoon_response', 'medium_multi_front', 'hard_cascading_crisis']
for task_id in tasks:
    result = run_task(env_core, task_id)
    print(f'{task_id}: score={result["score"]:.4f}, steps={result["steps_taken"]}')